# SpaceX Falcon 9 Launches: SQL Analysis

This notebook answers the ten SQL tasks from the lab on the SpaceX launch dataset. The data is loaded into a local SQLite database, the same approach the lab uses.

**Cleaning applied before loading:** text columns are trimmed, because some categories contain trailing spaces (for example `Success ` and `No attempt `). Rows with a missing date are dropped, which removes no rows here.

## 1. Load the data into SQLite

In [1]:
import os, sqlite3
import pandas as pd
from IPython.display import display

DATA = "/home/claude/spacex/data/Spacex.csv"
OUT = "/home/claude/spacex/outputs/sql"
os.makedirs(OUT, exist_ok=True)
DB = "/home/claude/spacex/data/spacex.db"
if os.path.exists(DB): os.remove(DB)

raw = pd.read_csv(DATA)
for c in raw.select_dtypes(include="object").columns:
    raw[c] = raw[c].str.strip()
raw = raw[raw["Date"].notna()]

con = sqlite3.connect(DB)
raw.to_sql("SPACEXTABLE", con, index=False)
print("rows loaded:", pd.read_sql("SELECT COUNT(*) AS n FROM SPACEXTABLE", con).n[0])
pd.read_sql("SELECT * FROM SPACEXTABLE LIMIT 3", con)

rows loaded: 101


/tmp/ipykernel_172/2207923683.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for c in raw.select_dtypes(include="object").columns:


,Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
0,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt


### Task 1

**Display the names of the unique launch sites in the space mission.**

In [2]:
q1 = """SELECT DISTINCT Launch_Site FROM SPACEXTABLE ORDER BY Launch_Site;"""
res1 = pd.read_sql(q1, con)
res1.to_csv(f"{OUT}/task1.csv", index=False)
display(res1)

,Launch_Site
0,CCAFS LC-40
1,CCAFS SLC-40
2,KSC LC-39A
3,VAFB SLC-4E


### Task 2

**Display 5 records where launch sites begin with the string 'CCA'.**

In [3]:
q2 = """SELECT * FROM SPACEXTABLE WHERE Launch_Site LIKE 'CCA%' LIMIT 5;"""
res2 = pd.read_sql(q2, con)
res2.to_csv(f"{OUT}/task2.csv", index=False)
display(res2)

,Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
0,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


### Task 3

**Display the total payload mass carried by boosters launched by NASA (CRS).**

In [4]:
q3 = """SELECT SUM(PAYLOAD_MASS__KG_) AS total_payload_kg FROM SPACEXTABLE WHERE Customer = 'NASA (CRS)';"""
res3 = pd.read_sql(q3, con)
res3.to_csv(f"{OUT}/task3.csv", index=False)
display(res3)

,total_payload_kg
0,45596


### Task 4

**Display the average payload mass carried by booster version F9 v1.1.**

In [5]:
q4 = """SELECT AVG(PAYLOAD_MASS__KG_) AS avg_payload_kg FROM SPACEXTABLE WHERE Booster_Version LIKE 'F9 v1.1%';"""
res4 = pd.read_sql(q4, con)
res4.to_csv(f"{OUT}/task4.csv", index=False)
display(res4)

,avg_payload_kg
0,2534.666667


### Task 5

**List the date when the first successful landing outcome on a ground pad was achieved.**

In [6]:
q5 = """SELECT MIN(Date) AS first_ground_pad_success FROM SPACEXTABLE WHERE Landing_Outcome = 'Success (ground pad)';"""
res5 = pd.read_sql(q5, con)
res5.to_csv(f"{OUT}/task5.csv", index=False)
display(res5)

,first_ground_pad_success
0,2015-12-22


### Task 6

**List the names of the boosters which have success in drone ship and have payload mass greater than 4000 but less than 6000.**

In [7]:
q6 = """SELECT Booster_Version FROM SPACEXTABLE WHERE Landing_Outcome = 'Success (drone ship)' AND PAYLOAD_MASS__KG_ > 4000 AND PAYLOAD_MASS__KG_ < 6000 ORDER BY Booster_Version;"""
res6 = pd.read_sql(q6, con)
res6.to_csv(f"{OUT}/task6.csv", index=False)
display(res6)

,Booster_Version
0,F9 FT B1021.2
1,F9 FT B1031.2
2,F9 FT B1022
3,F9 FT B1026


### Task 7

**List the total number of successful and failure mission outcomes.**

In [8]:
q7 = """SELECT Mission_Outcome, COUNT(*) AS total FROM SPACEXTABLE GROUP BY Mission_Outcome ORDER BY total DESC;"""
res7 = pd.read_sql(q7, con)
res7.to_csv(f"{OUT}/task7.csv", index=False)
display(res7)

,Mission_Outcome,total
0,Success,99
1,Success (payload status unclear),1
2,Failure (in flight),1


### Task 8

**List all booster versions that have carried the maximum payload mass, using a subquery with an aggregate function.**

In [9]:
q8 = """SELECT Booster_Version, PAYLOAD_MASS__KG_ FROM SPACEXTABLE WHERE PAYLOAD_MASS__KG_ = (SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTABLE) ORDER BY Booster_Version;"""
res8 = pd.read_sql(q8, con)
res8.to_csv(f"{OUT}/task8.csv", index=False)
display(res8)

,Booster_Version,PAYLOAD_MASS__KG_
0,F9 B5 B1048.4,15600
1,F9 B5 B1048.5,15600
2,F9 B5 B1049.4,15600
3,F9 B5 B1049.5,15600
4,F9 B5 B1049.7,15600
5,F9 B5 B1051.3,15600
6,F9 B5 B1051.4,15600
7,F9 B5 B1051.6,15600
8,F9 B5 B1056.4,15600
9,F9 B5 B1058.3,15600


### Task 9

**List the month, failed drone-ship landings, booster versions and launch site for the months in year 2015.**

SQLite has no month-name function, so the month is taken from the `Date` text with `substr`.

In [10]:
q9 = """SELECT substr(Date, 6, 2) AS month, Landing_Outcome, Booster_Version, Launch_Site FROM SPACEXTABLE WHERE substr(Date, 1, 4) = '2015' AND Landing_Outcome = 'Failure (drone ship)' ORDER BY month;"""
res9 = pd.read_sql(q9, con)
res9.to_csv(f"{OUT}/task9.csv", index=False)
display(res9)

,month,Landing_Outcome,Booster_Version,Launch_Site
0,01,Failure (drone ship),F9 v1.1 B1012,CCAFS LC-40
1,04,Failure (drone ship),F9 v1.1 B1015,CCAFS LC-40


### Task 10

**Rank the count of landing outcomes between 2010-06-04 and 2017-03-20, in descending order.**

In [11]:
q10 = """SELECT Landing_Outcome, COUNT(*) AS outcome_count FROM SPACEXTABLE WHERE Date BETWEEN '2010-06-04' AND '2017-03-20' GROUP BY Landing_Outcome ORDER BY outcome_count DESC;"""
res10 = pd.read_sql(q10, con)
res10.to_csv(f"{OUT}/task10.csv", index=False)
display(res10)

,Landing_Outcome,outcome_count
0,No attempt,10
1,Success (drone ship),5
2,Failure (drone ship),5
3,Success (ground pad),3
4,Controlled (ocean),3
5,Uncontrolled (ocean),2
6,Failure (parachute),2
7,Precluded (drone ship),1


## 2. Summary

In [12]:
con.close()
print("Saved", len(os.listdir(OUT)), "result files to", OUT)

Saved 10 result files to /home/claude/spacex/outputs/sql
